# Progress Review 1 — Kafka to MinIO Consumer

This notebook consumes real aviation weather events from Kafka topic `weather.raw` and writes them into MinIO object storage.

Flow:

Kafka topic `weather.raw` → Python Kafka consumer → MinIO raw bucket → optional Delta Lake table.

This proves the Progress Review 1 requirement: data flowing end-to-end through ingestion and storage.

In [1]:
import os
import json
import time
import uuid
from datetime import datetime, timezone
from io import BytesIO

import boto3
import pandas as pd
from kafka import KafkaConsumer

In [2]:
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS")
KAFKA_TOPIC = os.getenv("KAFKA_TOPIC_WEATHER_RAW")

MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT_INTERNAL")
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

RAW_BUCKET = os.getenv("RAW_BUCKET")
LAKEHOUSE_BUCKET = os.getenv("LAKEHOUSE_BUCKET")

required = {
    "KAFKA_BOOTSTRAP_SERVERS": KAFKA_BOOTSTRAP_SERVERS,
    "KAFKA_TOPIC_WEATHER_RAW": KAFKA_TOPIC,
    "MINIO_ENDPOINT_INTERNAL": MINIO_ENDPOINT,
    "MINIO_ROOT_USER": MINIO_ACCESS_KEY,
    "MINIO_ROOT_PASSWORD": MINIO_SECRET_KEY,
    "RAW_BUCKET": RAW_BUCKET,
    "LAKEHOUSE_BUCKET": LAKEHOUSE_BUCKET,
}

missing = [k for k, v in required.items() if not v]
if missing:
    raise RuntimeError(f"Missing environment variables: {missing}")

print("Kafka:", KAFKA_BOOTSTRAP_SERVERS)
print("Topic:", KAFKA_TOPIC)
print("MinIO:", MINIO_ENDPOINT)
print("Raw bucket:", RAW_BUCKET)
print("Lakehouse bucket:", LAKEHOUSE_BUCKET)

Kafka: kafka:9092
Topic: weather.raw
MinIO: http://minio:9000
Raw bucket: raw
Lakehouse bucket: lakehouse


In [3]:
s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name=AWS_REGION,
)

buckets = [b["Name"] for b in s3.list_buckets()["Buckets"]]
print("Buckets:", buckets)

for bucket in [RAW_BUCKET, LAKEHOUSE_BUCKET]:
    if bucket not in buckets:
        raise RuntimeError(f"Bucket missing: {bucket}")

Buckets: ['lakehouse', 'mlflow', 'raw', 'warehouse']


In [4]:
def utc_now_string():
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")


def write_jsonl_batch_to_minio(records, run_id, part_number):
    now = datetime.now(timezone.utc)

    key = (
        f"kafka_weather_events/"
        f"run_id={run_id}/"
        f"year={now.year}/"
        f"month={now.month:02d}/"
        f"day={now.day:02d}/"
        f"part-{part_number:05d}.jsonl"
    )

    body = "\n".join(json.dumps(record, default=str) for record in records) + "\n"

    s3.put_object(
        Bucket=RAW_BUCKET,
        Key=key,
        Body=body.encode("utf-8"),
        ContentType="application/json",
    )

    print(f"Wrote JSONL batch: s3://{RAW_BUCKET}/{key}")
    return key


def flatten_for_delta(records):
    clean_records = []

    for record in records:
        clean = {}
        for key, value in record.items():
            if isinstance(value, (dict, list)):
                clean[key] = json.dumps(value)
            else:
                clean[key] = value

        if "timestamp_utc" in clean:
            clean["timestamp_utc"] = str(clean["timestamp_utc"])

        if "_ingested_at_utc" in clean:
            clean["_ingested_at_utc"] = str(clean["_ingested_at_utc"])

        clean_records.append(clean)

    return clean_records


def write_delta_batch_to_minio(records):
    from deltalake import write_deltalake

    clean_records = flatten_for_delta(records)
    df = pd.DataFrame(clean_records)

    storage_options = {
        "AWS_ENDPOINT_URL": MINIO_ENDPOINT,
        "AWS_ACCESS_KEY_ID": MINIO_ACCESS_KEY,
        "AWS_SECRET_ACCESS_KEY": MINIO_SECRET_KEY,
        "AWS_REGION": AWS_REGION,
        "AWS_ALLOW_HTTP": "true",
        "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
    }

    table_uri = f"s3://{LAKEHOUSE_BUCKET}/bronze/weather_events_delta"

    write_deltalake(
        table_uri,
        df,
        mode="append",
        storage_options=storage_options,
    )

    print(f"Appended {len(df)} rows to Delta table: {table_uri}")

In [5]:
RUN_ID = utc_now_string() + "-" + str(uuid.uuid4())[:8]

# Your downloaded sample has 72 rows: 3 days × 24 hours.
MAX_RECORDS = 72

FLUSH_SIZE = 10

# Keep this True. If Delta fails, JSONL raw storage will still work.
WRITE_DELTA = True

CONSUMER_GROUP_ID = f"pr1-minio-sink-{RUN_ID}"

print("Run ID:", RUN_ID)
print("Consumer group:", CONSUMER_GROUP_ID)
print("Max records:", MAX_RECORDS)
print("Flush size:", FLUSH_SIZE)
print("Write Delta:", WRITE_DELTA)

Run ID: 20260425T094031Z-d767cd57
Consumer group: pr1-minio-sink-20260425T094031Z-d767cd57
Max records: 72
Flush size: 10
Write Delta: True


In [6]:
consumer = KafkaConsumer(
    KAFKA_TOPIC,
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    group_id=CONSUMER_GROUP_ID,
    auto_offset_reset="latest",
    enable_auto_commit=True,
    value_deserializer=lambda b: json.loads(b.decode("utf-8")),
    consumer_timeout_ms=1000,
)

print("Consumer started.")
print("Now run Notebook 03 producer in another Jupyter tab.")
print("Waiting for new Kafka messages...")

buffer = []
stored_keys = []
total = 0
part_number = 0
idle_loops = 0

while True:
    message_seen = False

    for message in consumer:
        message_seen = True

        event = message.value
        event["_kafka_topic"] = message.topic
        event["_kafka_partition"] = message.partition
        event["_kafka_offset"] = message.offset
        event["_ingested_at_utc"] = datetime.now(timezone.utc).isoformat()

        buffer.append(event)
        total += 1

        print(
            f"received={total:03d} "
            f"offset={message.offset} "
            f"airport={event.get('airport')} "
            f"time={event.get('timestamp_utc')} "
            f"wind_kts={event.get('wind_speed_kts')}"
        )

        if len(buffer) >= FLUSH_SIZE:
            part_number += 1

            key = write_jsonl_batch_to_minio(
                records=buffer,
                run_id=RUN_ID,
                part_number=part_number,
            )
            stored_keys.append(key)

            if WRITE_DELTA:
                try:
                    write_delta_batch_to_minio(buffer)
                except Exception as exc:
                    print("Delta write failed, but JSONL storage succeeded.")
                    print("Delta error:", repr(exc))

            buffer = []

        if total >= MAX_RECORDS:
            break

    if total >= MAX_RECORDS:
        break

    if not message_seen:
        idle_loops += 1
        time.sleep(1)
    else:
        idle_loops = 0

    if idle_loops >= 60:
        print("No new messages for 60 seconds. Stopping consumer.")
        break

if buffer:
    part_number += 1

    key = write_jsonl_batch_to_minio(
        records=buffer,
        run_id=RUN_ID,
        part_number=part_number,
    )
    stored_keys.append(key)

    if WRITE_DELTA:
        try:
            write_delta_batch_to_minio(buffer)
        except Exception as exc:
            print("Delta write failed, but JSONL storage succeeded.")
            print("Delta error:", repr(exc))

consumer.close()

print("\nConsumer finished.")
print("Total records consumed:", total)
print("Stored JSONL objects:")
for key in stored_keys:
    print(f"  s3://{RAW_BUCKET}/{key}")
print("Run ID:", RUN_ID)

Consumer started.
Now run Notebook 03 producer in another Jupyter tab.
Waiting for new Kafka messages...
received=001 offset=0 airport=JFK time=2022-01-01 00:00:00 wind_kts=3.638605833053589
received=002 offset=1 airport=JFK time=2022-01-01 01:00:00 wind_kts=3.977353811264038
received=003 offset=2 airport=JFK time=2022-01-01 02:00:00 wind_kts=4.554651737213135
received=004 offset=3 airport=JFK time=2022-01-01 03:00:00 wind_kts=4.152695178985596
received=005 offset=4 airport=JFK time=2022-01-01 04:00:00 wind_kts=4.710644721984863
received=006 offset=5 airport=JFK time=2022-01-01 05:00:00 wind_kts=4.531341552734375
received=007 offset=6 airport=JFK time=2022-01-01 06:00:00 wind_kts=4.341426849365234
received=008 offset=7 airport=JFK time=2022-01-01 07:00:00 wind_kts=4.927023887634277
received=009 offset=8 airport=JFK time=2022-01-01 08:00:00 wind_kts=4.280397415161133
received=010 offset=9 airport=JFK time=2022-01-01 09:00:00 wind_kts=5.644429683685303
Wrote JSONL batch: s3://raw/kafka_w